In [ ]:
import kagglehub
import os
import csv
import json
import math
import random
from collections import defaultdict

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andreagarritano/twitch-social-networks")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'twitch-social-networks' dataset.
Path to dataset files: /kaggle/input/twitch-social-networks


Set the file paths for ENGB dataser

In [ ]:
base_path = os.path.join(path, "ENGB")

edges_file = os.path.join(base_path, "ENGB_edges.csv")
features_file = os.path.join(base_path, "ENGB_features.json")
targets_file = os.path.join(base_path, "ENGB_target.csv")

Step 2: Load the edges

In [ ]:
edges = []
all_nodes = set()

with open(edges_file, "r", newline="", encoding="utf-8") as f:
  reader = csv.DictReader(f)
  for row in reader:
    u = int(row["from"])
    v = int(row["to"])
    edges.append((u, v))
    all_nodes.add(u)
    all_nodes.add(v)

print("Total edges:", len(edges))
print("Total nodes so far:", len(all_nodes))

Total edges: 35324
Total nodes so far: 7126


Step 3: Build the graph in NetworkX

In [ ]:
import networkx as nx

G = nx.Graph()

for u, v in edges:
  G.add_edge(u, v)

print("Number of nodes:", len(G.nodes))
print("Number of edges:", len(G.edges))


Number of nodes: 7126
Number of edges: 35324


Step 4: Create a consistent node index mappinhg

In [ ]:
nodes = sorted(G.nodes())

node_to_idx = {}
idx_to_node = {}

for i, node in enumerate(nodes):
  node_to_idx[node] = i
  idx_to_node[i] = node

N = len(nodes)

print("Total nodes:", N)
print("First 5 nodes:", nodes[:5])
print("Example mapping:", nodes[0], "->", node_to_idx[nodes[0]])

Total nodes: 7126
First 5 nodes: [0, 1, 2, 3, 4]
Example mapping: 0 -> 0


Step 5: Build A1 which is the direct friendship adjacency matrix

In [ ]:
A1 = [[0 for _ in range(N)] for _ in range(N)]

for u, v in G.edges():
  i = node_to_idx[u]
  j = node_to_idx[v]

  A1[i][j] = 1
  A1[j][i] = 1 #because the graph is undirected

print("A1 built successfully")
print("Matrix size", len(A1), "x", len(A1[0]))

A1 built successfully
Matrix size 7126 x 7126


Step 6: Normalize A1

In [ ]:
for i in range(N):
  A1[i][i] = 1

degree = [sum(A1[i]) for i in range(N)]

A1_norm = [[0 for _ in range(N)] for _ in range(N)]

for i in range(N):
  for j in range(N):
    if A1[i][j] !=0:
      A1_norm[i][j] = A1[i][j] / math.sqrt(degree[i] * degree[j])

Step 7: Build A2 which is the shared friends adjacency matrix
A2[i][j] = how many friends users i and j have in common
computed by multipying A1 with itself (2-hop neighborhood)

In [ ]:
A2 = [[0 for _ in range(N)] for _ in range(N)]

# precompute neighbors (speeds things up)
neighbors = [set() for _ in range(N)]

for i in range(N):
  for j in range(N):
    if A1[i][j] == 1:
        neighbors[i].add(j)

for i in range(N):
    for j in range(i + 1, N):
        common = len(neighbors[i].intersection(neighbors[j]))

    if common >= 3 and A1[i][j] == 0:
        A2[i][j] = 1
        A2[j][i] = 1

print("A2 built successfully")
print("Matrix size", len(A2), "x", len(A2[0]))

A2 built successfully
Matrix size 7126 x 7126


In [ ]:
# add self-loops

for i in range(N):
    A2[i][i] = 1

# compute degrees
degree2 = [sum(A2[i]) for i in range(N)]

#normalize
A2_norm = [[0 for _ in range(N)] for _ in range(N)]

for i in range(N):
    for j in range(N):
        if A2[i][j] != 0:
            A2_norm[i][j] = A2[i][j] / math.sqrt(degree2[i] * degree2[j])

print("A2 normalized")

A2 normalized


Step 8: Load the features and make sure to build the feature matrix X

In [ ]:
import json

with open(os.path.join(path, "ENGB", "ENGB_features.json"), "r") as f:
    features_raw = json.load(f)

print(type(features_raw))
print(list(features_raw.items()) [:2])

<class 'dict'>
[('3032', [2605, 1191, 357, 2120, 861, 231, 3164, 920, 1907, 1612, 2327, 2549, 2524, 2912, 139, 83, 276, 2645, 2424, 2501, 2737, 1525, 751]), ('4032', [1521, 1191, 2334, 846, 3103, 3045, 920, 224, 810, 1369, 569, 2384, 2362, 1762, 3106, 751])]


In [ ]:
#find total number of unique features
all_features = set()

for feat_list in features_raw.values():
    all_features.update(feat_list)

F = max(all_features) + 1

print("Nodes:", N)
print("Features:", F)

# build dense feature matrix

X = [[0 for _ in range(F)] for _ in range(N)]

for node_id, feat_list in features_raw.items():
  node = int(node_id)

  if node in node_to_idx:
        idx = node_to_idx[node]

        for feat in feat_list:
            X[idx][feat] = 1

print("X built")
print("Rows:", len(X))
print("Cols:", len(X[0]))  # total number of active features across all users
print("First row sample:", X[0][:20])

Nodes: 7126
Features: 3170
X built
Rows: 7126
Cols: 3170
First row sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
feature_sets = []

for i in range(N):
  current_set = set()
  for j in range(F):
    if X[i][j] == 1:
        current_set.add(j)
  feature_sets.append(current_set)

print("Feature sets built")

Feature sets built


In [ ]:
A3 = [[0 for _ in range(N)] for _ in range(N)]

for i in range(N):
  for j in range(i +1, N):
      common = len(feature_sets[i].intersection(feature_sets[j]))

      if len(feature_sets[i]) > 0 and len(feature_sets[j]) > 0:
        cosine_sim = common / math.sqrt(len(feature_sets[i]) * len(feature_sets[j]))

        if cosine_sim >= 0.5:
          A3[i][j] = 1
          A3[j][i] = 1

print("A3 built successfully")
print("Matrix size:", len(A3), "x", len(A3[0]))


A3 built successfully
Matrix size: 7126 x 7126


In [ ]:
for i in range(N):
  A3[i][i] = 1

degree3 = [sum(A3[i]) for i in range(N)]

A3_norm = [[0 for _ in range(N)] for _ in range(N)]

for i in range(N):
  for j in range(N):
    if A3[i][j] != 0:
        A3_norm[i][j] = A3[i][j] / math.sqrt(degree3[i] * degree3[j])

print("A3 normalized")
print("First row sample:", A3_norm[0][:10])


A3 normalized
First row sample: [0.3333333333333333, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
for i in range(N):
  A3[i][i] = 0

print("A3 built successfully")
print("Matrix size:", len(A3), "x", len(A3[0]))

A3 built successfully
Matrix size: 7126 x 7126


In [ ]:
for i in range(N):
  A3[i][i] = 1

degree3 = [sum(A3[i]) for i in range(N)]

A3_norm = [[0 for _ in range(N)] for _ in range(N)]

for i in range(N):
  for j in range(N):
    if A3[i][j] !=0:
      A3_norm[i][j] = A3[i][j] / math.sqrt(degree3[i] * degree3[j])

print("A3 normalized")
print("First row sample:", A3_norm[0][:10])

A3 normalized
First row sample: [0.3333333333333333, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Step 10: Load labels the task is predicting whether a streamer has mature content True or False

In [ ]:
targets = []

with open(os.path.join(path, "ENGB", "ENGB_target.csv"), "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
      targets.append(row)

print("Loaded targets:", len(targets))
print("Sample row:", targets[0])

Loaded targets: 7126
Sample row: {'id': '73045350', 'days': '1459', 'mature': 'False', 'views': '9528', 'partner': 'False', 'new_id': '2299'}


This builds the label vector

In [ ]:
y = [0 for _ in range(N)]

for row in targets:
    node_id = int(row["new_id"])

    if node_id in node_to_idx:
      idx = node_to_idx[node_id]
      y[idx] = 1 if row["mature"] == "True" else 0

Step 11: This converts everything to PyTorch tensors

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
X_tensor = torch.FloatTensor(X)
A1_tensor = torch.FloatTensor(A1_norm)
A2_tensor = torch.FloatTensor(A2_norm)
A3_tensor = torch.FloatTensor(A3_norm)
y_tensor = torch.FloatTensor(y)

adj_list = [A1_tensor, A2_tensor, A3_tensor]

print(X_tensor.shape)
print(y_tensor.shape)
print(len(adj_list))

torch.Size([7126, 3170])
torch.Size([7126])
3


In [ ]:
torch.manual_seed(42)

indices = torch.randperm(N)

train_size = int(0.7 * N)
val_size = int(0.1 * N)
test_size = N - train_size - val_size

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

print("Train size:", len(train_idx))
print("Validation size:", len(val_idx))
print("Test size:", len(test_idx))

Train size: 4988
Validation size: 712
Test size: 1426


In [ ]:
class MGCNLayer(nn.Module):
  def __init__(self, in_features, out_features):
    super(MGCNLayer, self).__init__()

    self.W1 = nn.Linear(in_features, out_features, bias=False)
    self.W2 = nn.Linear(in_features, out_features, bias=False)
    self.W3 = nn.Linear(in_features, out_features, bias=False)

    self.combine = nn.Linear(out_features * 3, out_features)

  def forward(self, X, adj_list):
      A1, A2, A3 = adj_list

      H1 = self.W1(torch.matmul(A1, X))
      H2 = self.W2(torch.matmul(A2, X))
      H3 = self.W3(torch.matmul(A3, X))

      H = torch.cat([H1, H2, H3], dim=1)
      H = self.combine(H)

      return H

In [ ]:
class MultiGraphGCN(nn.Module):
  def __init__(self, in_features, hidden_features, dropout=0.5):
    super(MultiGraphGCN, self).__init__()

    self.layer1 = MGCNLayer(in_features, hidden_features)
    self.layer2 = MGCNLayer(hidden_features, hidden_features)
    self.out = nn.Linear(hidden_features, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, X, adj_list):
    H = self.layer1(X, adj_list)
    H = F.relu(H)
    H = self.dropout(H)

    H = self.layer2(H, adj_list)
    H = F.relu(H)
    H = self.dropout(H)

    logits = self.out(H).squeeze(1)
    return logits

In [ ]:
model = MultiGraphGCN(
    in_features= X_tensor.shape[1],
    hidden_features=32,
    dropout=0.5
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

print(model)

MultiGraphGCN(
  (layer1): MGCNLayer(
    (W1): Linear(in_features=3170, out_features=32, bias=False)
    (W2): Linear(in_features=3170, out_features=32, bias=False)
    (W3): Linear(in_features=3170, out_features=32, bias=False)
    (combine): Linear(in_features=96, out_features=32, bias=True)
  )
  (layer2): MGCNLayer(
    (W1): Linear(in_features=32, out_features=32, bias=False)
    (W2): Linear(in_features=32, out_features=32, bias=False)
    (W3): Linear(in_features=32, out_features=32, bias=False)
    (combine): Linear(in_features=96, out_features=32, bias=True)
  )
  (out): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
def binary_accuracy(logits, labels):
  preds = (torch.sigmoid(logits) >= 0.5).float()
  correct = (preds == labels).float().mean().item()
  return correct

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):
  model.train()
  optimizer.zero_grad()

  logits = model(X_tensor, adj_list)

  train_logits = logits[train_idx]
  train_labels = y_tensor[train_idx]

  loss = criterion(train_logits, train_labels)
  loss.backward()
  optimizer.step()

  model.eval()
  with torch.no_grad():
    logits = model(X_tensor, adj_list)

    train_loss = criterion(logits[train_idx], y_tensor[train_idx]).item()
    val_loss = criterion(logits[val_idx], y_tensor[val_idx]).item()

    train_acc = binary_accuracy(logits[train_idx], y_tensor[train_idx])
    val_acc = binary_accuracy(logits[val_idx], y_tensor[val_idx])

  if epoch % 10 == 0:
    print("Epoch", epoch,
          "| Train Loss =", round(train_loss, 4),
          "| Val Loss =", round(val_loss, 4),
          "| Train Acc =", round(train_acc, 4),
          "| Val Acc =", round(val_acc, 4))

Epoch 0 | Train Loss = 0.6881 | Val Loss = 0.6897 | Train Acc = 0.5473 | Val Acc = 0.5379
Epoch 10 | Train Loss = 0.5952 | Val Loss = 0.6672 | Train Acc = 0.6975 | Val Acc = 0.618
Epoch 20 | Train Loss = 0.4305 | Val Loss = 0.8634 | Train Acc = 0.8021 | Val Acc = 0.5983
Epoch 30 | Train Loss = 0.3134 | Val Loss = 1.0598 | Train Acc = 0.8623 | Val Acc = 0.5787
Epoch 40 | Train Loss = 0.2173 | Val Loss = 1.5341 | Train Acc = 0.9014 | Val Acc = 0.5716
Epoch 50 | Train Loss = 0.2509 | Val Loss = 1.712 | Train Acc = 0.8887 | Val Acc = 0.5562
Epoch 60 | Train Loss = 0.1089 | Val Loss = 2.9011 | Train Acc = 0.9533 | Val Acc = 0.559
Epoch 70 | Train Loss = 0.0811 | Val Loss = 3.8845 | Train Acc = 0.9671 | Val Acc = 0.566
Epoch 80 | Train Loss = 0.3184 | Val Loss = 1.2399 | Train Acc = 0.8739 | Val Acc = 0.5506
Epoch 90 | Train Loss = 0.2093 | Val Loss = 1.6548 | Train Acc = 0.9266 | Val Acc = 0.5688


In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X_tensor, adj_list)

    test_logits = logits[test_idx]
    test_labels = y_tensor[test_idx]

    test_loss = criterion(test_logits, test_labels).item()
    test_acc = binary_accuracy(test_logits, test_labels)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy", round(test_acc, 4))



Test Loss: 2.6439
Test Accuracy 0.5421


In [ ]:
with torch.no_grad():
  probs = torch.sigmoid(logits[test_idx])

for i in range(10):
  predicted_prob = probs[i].item()
  predicted_label = 1 if predicted_prob >= 0.5 else 0
  actual_label = int(test_labels[i].item())

  print("Predicted probability:", round(precited_prob, 4)
        "| Predicted label:" predicted_label,
        "| Actual label:", actual_label)

In [ ]:
with torch.no_grad():
  final_preds = (torch.sigmoid(logits[test_idx]) >= 0.5).float()

print("Predicted positives:", int(final_preds.sum().item()))
print("Predicted negatives:", int(len(final_preds) - final_preds.sum(),item()))


Up next is stacking the two layers and output and then training and test split the data with a training loop

In [ ]:
for root, dirs, files in os.walk(path):
  for f in files:
    print(os.path.join(root,f))

/kaggle/input/twitch-social-networks/ES/ES_target.csv
/kaggle/input/twitch-social-networks/ES/ES_edges.csv
/kaggle/input/twitch-social-networks/ES/ES_features.json
/kaggle/input/twitch-social-networks/PTBR/PTBR_target.csv
/kaggle/input/twitch-social-networks/PTBR/PTBR_features.json
/kaggle/input/twitch-social-networks/PTBR/PTBR_edges.csv
/kaggle/input/twitch-social-networks/RU/RU_features.json
/kaggle/input/twitch-social-networks/RU/RU_edges.csv
/kaggle/input/twitch-social-networks/RU/RU_target.csv
/kaggle/input/twitch-social-networks/FR/FR_target.csv
/kaggle/input/twitch-social-networks/FR/FR_features.json
/kaggle/input/twitch-social-networks/FR/FR_edges.csv
/kaggle/input/twitch-social-networks/DE/DE_target.csv
/kaggle/input/twitch-social-networks/DE/DE.json
/kaggle/input/twitch-social-networks/DE/DE_edges.csv
/kaggle/input/twitch-social-networks/ENGB/ENGB_target.csv
/kaggle/input/twitch-social-networks/ENGB/ENGB_features.json
/kaggle/input/twitch-social-networks/ENGB/ENGB_edges.csv
